# Использование графов знаний в машинном обучении

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* https://www.dgl.ai/
* https://docs.dgl.ai/en/latest/guide/index.html

## Задачи для самостоятельного решения

<p class="task" id="1"></p>

1\. Используя информацию из Wikidata, сгенерируйте набор данных из литературных произведений, написанных с 1700 по 2000 год. Для каждого произведения получите:
* название;
* жанр;
* автора;
* дату публикации;
* основную тему;
* дату создания;
* форму творческой работы.

Для всех авторов, который оказались в выборке, получите:
* пол;
* место рождения;
* список людей, которыми вдохновлялся автор.
  
Представьте результаты в виде `pd.DataFrame`.

- [ ] Проверено на семинаре

In [2]:
!pip install SPARQLWrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 8.6 MB/s eta 0:00:00


In [3]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
from datetime import datetime
from typing import List, Tuple, Dict, Any
import time

In [8]:
def build_optimized_works_sparql(year_from: int = 1700, year_to: int = 2000, limit: int = 1000) -> str:
    """
    Один запрос: возвращает произведения + агрегированные данные по жанрам и авторам.
    Агрегируем авторские поля (label, uri, gender, birthplace, influenced_by) через GROUP_CONCAT.
    Это уменьшает число запросов к endpoint.
    """
    start = f"{year_from}-01-01T00:00:00Z"
    end = f"{year_to}-12-31T23:59:59Z"
    q = f"""
    PREFIX wd: <http://www.wikidata.org/entity/>
    PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

    SELECT ?work
           (SAMPLE(?workLabel) AS ?title)
           (GROUP_CONCAT(DISTINCT ?genreLabel; separator="||") AS ?genres)
           (GROUP_CONCAT(DISTINCT ?authorLabel; separator="||") AS ?author_labels)
           (GROUP_CONCAT(DISTINCT ?author; separator="||") AS ?author_uris)
           (GROUP_CONCAT(DISTINCT ?genderLabel; separator="||") AS ?author_genders)
           (GROUP_CONCAT(DISTINCT ?birthPlaceLabel; separator="||") AS ?author_birthplaces)
           (GROUP_CONCAT(DISTINCT ?influencedByLabel; separator="||") AS ?author_influenced_by)
           (SAMPLE(?pubDate) AS ?pubDate)
           (SAMPLE(?mainSubjectLabel) AS ?main_subject)
           (SAMPLE(?inception) AS ?creation_date)
           (SAMPLE(?formLabel) AS ?form_of_creative_work)
    WHERE {{
      # Ограничиваем типы произведений: все подклассы literary work (Q7725634)
      ?work wdt:P31/wdt:P279* wd:Q7725634 .
      ?work wdt:P577 ?pubDate .
      FILTER (?pubDate >= "{start}"^^xsd:dateTime && ?pubDate <= "{end}"^^xsd:dateTime)

      OPTIONAL {{ ?work wdt:P136 ?genre. }}
      OPTIONAL {{
        ?work wdt:P50 ?author.
        OPTIONAL {{ ?author wdt:P21 ?gender. }}
        OPTIONAL {{ ?author wdt:P19 ?birthPlace. }}
        OPTIONAL {{ ?author wdt:P737 ?influencedBy. }}
      }}
      OPTIONAL {{ ?work wdt:P921 ?mainSubject. }}
      OPTIONAL {{ ?work wdt:P571 ?inception. }}
      OPTIONAL {{ ?work wdt:P7937 ?form. }}

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en".
        ?work rdfs:label ?workLabel .
        ?genre rdfs:label ?genreLabel .
        ?author rdfs:label ?authorLabel .
        ?gender rdfs:label ?genderLabel .
        ?birthPlace rdfs:label ?birthPlaceLabel .
        ?influencedBy rdfs:label ?influencedByLabel .
        ?mainSubject rdfs:label ?mainSubjectLabel .
        ?form rdfs:label ?formLabel .
      }}
    }}
    GROUP BY ?work
    ORDER BY ?pubDate
    LIMIT {limit}
    """
    return q

def run_sparql(endpoint: str, query: str, retries: int = 2, pause: float = 0.5, timeout: int = 60) -> List[Dict[str, Any]]:
    """
    Выполняем запрос. Минимум ретраев, небольшая пауза. Устанавливаем User-Agent.
    """
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    try:
        sparql.addCustomHeader("User-Agent", "WikidataDatasetBuilder/optimized (contact@example.com)")
    except Exception:
        pass
    # Попробуем установить таймаут (в секундах) — поддерживается в большинстве версий SPARQLWrapper
    try:
        sparql.setTimeout(timeout)
    except Exception:
        # Игнорируем, если метод недоступен
        pass

    attempt = 0
    while attempt <= retries:
        try:
            res = sparql.query().convert()
            return res.get("results", {}).get("bindings", [])
        except Exception as e:
            attempt += 1
            if attempt > retries:
                raise
            time.sleep(pause * attempt)
    return []

def bindings_to_dataframe_fast(bindings: List[Dict[str, Any]]) -> pd.DataFrame:
    """
    Быстрая конвертация bindings -> DataFrame.
    Преобразует значения 'value' и приводит даты к datetime.
    """
    if not bindings:
        return pd.DataFrame()

    # Выделим все ключи (всего немного в оптимизированном запросе)
    rows = []
    for b in bindings:
        row = {k: v.get("value") for k, v in b.items()}
        rows.append(row)
    df = pd.DataFrame(rows)

    # Приводим даты
    for col in ("pubDate", "creation_date"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")
    return df

def split_concat_field(s: str) -> List[str]:
    """Разделяет GROUP_CONCAT-поле; возвращает [] при пустом/None."""
    if not s:
        return []
    return [x for x in s.split("||") if x != ""]

def uri_to_qid(uri: str) -> str:
    if not uri:
        return None
    if "/" in uri:
        return uri.rsplit("/", 1)[-1]
    return uri

def explode_authors_from_works_df(works_df: pd.DataFrame) -> pd.DataFrame:
    """
    На входе works_df содержит агрегированные поля:
      author_uris, author_labels, author_genders, author_birthplaces, author_influenced_by
    Разворачиваем их в таблицу authors_df с одной строкой на (author_qid).
    Сопоставление выполняется по позиции; если позиции не совпадают — возможны несоответствия,
    но в практике групп-конкат обычно сохраняют согласованный порядок.
    """
    rows = []
    for _, r in works_df.iterrows():
        uris = split_concat_field(r.get("author_uris"))
        labels = split_concat_field(r.get("author_labels"))
        genders = split_concat_field(r.get("author_genders"))
        births = split_concat_field(r.get("author_birthplaces"))
        influenced = split_concat_field(r.get("author_influenced_by"))

        n = max(len(uris), len(labels), len(genders), len(births), len(influenced))
        # расширяем списки до длины n
        def extend(lst):
            return lst + [None] * (n - len(lst))
        uris, labels, genders, births, influenced = map(extend, (uris, labels, genders, births, influenced))

        for i in range(n):
            qid = uri_to_qid(uris[i]) if uris[i] else None
            rows.append({
                "author_qid": qid,
                "author_label": labels[i],
                "gender": genders[i],
                "birthplace": births[i],
                "influenced_by": influenced[i]  # это отдельная строка; для коллекции можно сгруппировать позже
            })
    # Создадим DataFrame и оставим уникальных авторов (по qid)
    authors_df = pd.DataFrame(rows)
    # Если хотим собрать influenced_by в список по автору:
    if not authors_df.empty:
        # Группируем и агрегируем influenced_by в список уникальных значений
        agg = authors_df.groupby("author_qid").agg({
            "author_label": "first",
            "gender": lambda x: next((v for v in x if pd.notna(v)), None),
            "birthplace": lambda x: next((v for v in x if pd.notna(v)), None),
            "influenced_by": lambda s: [v for v in pd.unique([v for v in s if pd.notna(v)])]
        }).reset_index()
        return agg
    else:
        return pd.DataFrame(columns=["author_qid", "author_label", "gender", "birthplace", "influenced_by"])

def build_dataset_optimized(year_from: int = 1700, year_to: int = 2000, limit: int = 500, sparql_timeout: int = 60) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Основная оптимизированная функция. Возвращает (works_df, authors_df).
    works_df содержит агрегированные поля (genres, author_*), authors_df — список авторов с полем influenced_by как список.
    """
    query = build_optimized_works_sparql(year_from, year_to, limit)
    bindings = run_sparql(WIKIDATA_SPARQL, query, retries=2, pause=0.4, timeout=sparql_timeout)
    works_df = bindings_to_dataframe_fast(bindings)

    # Переименуем колонки в более удобные
    rename_map = {
        "title": "title",
        "genres": "genres",
        "author_labels": "author_labels",
        "author_uris": "author_uris",
        "author_genders": "author_genders",
        "author_birthplaces": "author_birthplaces",
        "author_influenced_by": "author_influenced_by",
        "pubDate": "publication_date",
        "main_subject": "main_subject",
        "creation_date": "creation_date",
        "form_of_creative_work": "form_of_creative_work"
    }
    # оставляем только ожидаемые столбцы (если они есть)
    works_df = works_df.rename(columns=rename_map)

    # Преобразуем агрегированные поля в списки (lazy — только при необходимости)
    for col in ("genres", "author_labels", "author_uris", "author_genders", "author_birthplaces", "author_influenced_by"):
        if col in works_df.columns:
            works_df[col] = works_df[col].apply(split_concat_field)

    # Создаём authors_df, разворачивая агрегированные author_* поля
    authors_df = explode_authors_from_works_df(works_df)

    # Для удобства: разбиваем genres в строку (или оставляем как список)
    if "genres" in works_df.columns:
        works_df["genres"] = works_df["genres"].apply(lambda x: x if isinstance(x, list) else split_concat_field(x))

    # Отбрасываем сырые агрегатные колонки, если нужно — можно оставить
    # но для полной информации оставим их. Пользователь может затем merge по author_qid.
    return works_df, authors_df

In [ ]:
# меньше limit = быстрее; подберите разумный лимит (например, 200-1000)
works_df, authors_df = build_dataset_optimized(1700, 2000, limit=500, sparql_timeout=60)
print("Works rows:", len(works_df))
print(works_df.head(3).to_dict(orient="records"))
print("Authors rows:", len(authors_df))
print(authors_df.head(10).to_dict(orient="records"))

<p class="task" id="2"></p>

2\. Полученный набор данных будет использоваться для решения задачи классификации книг по жанрам.

Проведите предобработку собранного датасета. Для таблицы с информацией о произведениях:
1. Сохраните в выборке только произведения, принадлежащие топ-20 жанрам (по частоте);
2. Для всех столбцов с датами извлеките век;
3. Все категориальные столбцы закодируйте при помощи `OrdinalEncoder` из sklearn;
4. Разделите на обучающее и тестовое множество с сохранением пропорций классов;

Выведите основную информацию о полученном датасете:
- количество строк в обучающем и тестовом множестве;
- количество уникальных значений в категориальных столбцах.

Таблицу с данными об авторами в этом и следующем задании игнорируйте.

- [ ] Проверено на семинаре

<p class="task" id="3"></p>

3\. Решите задачу классификации при помощи пакета `torch`. Архитектуру, функцию потерь, гиперпараметры модели и пр. выберите самостоятельно. Визуализируйте график функции потерь в зависимости от эпохи. Выведите на экран отчет по классификации для тестового множества.

- [ ] Проверено на семинаре

<p class="task" id="4"></p>

4\. Создайте гетерограф на основе собранных данных при помощи пакета `dgl`. В полученном графе должно быть два типа связей (человек, произведение) и четыре типа связей ("автор создал произведение", "произведение создано автором", "человек оказал влияние на другого человека", "человек испытал влияние другого человека"). Процедура предобработки признаков - аналогично заданию 2. Обратите внимание, что разбиение на обучающее и тестовое множество в данном случае заменяется на создание масок для обучающего/тестового множества, и данное разбиение должно соответствовать разбиению из задания 2.

Выведите на экран основую информацию о графе.

- [ ] Проверено на семинаре

<p class="task" id="5"></p>

5\. Используя созданный граф, решите задачу классификации узлов с использованием графовых нейронных сетей. Архитектуру, функцию потерь, гиперпараметры модели и пр. выберите самостоятельно. Визуализируйте график функции потерь в зависимости от эпохи. Выведите на экран отчет по классификации для тестового множества.

- [ ] Проверено на семинаре